In [1]:
import numpy as np
import pandas as pd
import torch
import yfinance as yf

from chronos import BaseChronosPipeline  # chronos-forecasting
from backtesting import Backtest, Strategy

/opt/miniconda3/envs/otus_new/lib/python3.12/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
# Данные
TICKER = "AAPL"
START = "2018-01-01"
END   = None

df = yf.download(TICKER, start=START, end=END, auto_adjust=False, progress=False)
df = df.dropna()

# backtesting.py ожидает колонки строго: Open High Low Close Volume
df = df[["Open", "High", "Low", "Close", "Volume"]].copy()
df.columns = ["Open", "High", "Low", "Close", "Volume"]

In [3]:
df.head()

,Open,High,Low,Close,Volume
Date,,,,,
2018-01-02,42.540001,43.075001,42.314999,43.064999,102223600
2018-01-03,43.132500,43.637501,42.990002,43.057499,118071600
2018-01-04,43.134998,43.367500,43.020000,43.257500,89738400
2018-01-05,43.360001,43.842499,43.262501,43.750000,94640000
2018-01-08,43.587502,43.902500,43.482498,43.587502,82271200


In [4]:
device_map = "cuda" if torch.cuda.is_available() else "cpu"  # можно заменить на "mps" для Mac

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-bolt-tiny",
    device_map=device_map,
    dtype=torch.bfloat16 if device_map != "cpu" else torch.float32,
)

In [5]:
# Rolling-прогноз: завтра (t+1)
# -----------------------
LOOKBACK = 256            # сколько последних дней даём модели как "контекст"
THRESH   = 0.002          # порог для сигнала (0.2%) чтобы отсечь шум
PRED_LEN = 1

close = df["Close"].values.astype(np.float32)

In [6]:
# найдём индекс квантиля 0.5 (медиана), если доступно
def get_median_forecast(forecast_tensor, pipeline_obj):
    # forecast_tensor shape: [1, Q, 1]
    Q = forecast_tensor.shape[1]
    q_list = getattr(pipeline_obj, "quantiles", None)
    if q_list is not None and 0.5 in q_list:
        qi = q_list.index(0.5)
    else:
        # fallback: берём центральный квантиль
        qi = Q // 2
    return float(forecast_tensor[0, qi, 0].detach().cpu().float().numpy())

In [7]:
signals = np.zeros(len(df), dtype=np.int8)   # -1 short, 0 flat, +1 long
pred_next = np.full(len(df), np.nan, dtype=np.float32)

In [8]:
# сигнал на день t рассчитываем по инфо до t включительно, торгуем на t+1
for t in range(LOOKBACK, len(df) - 1):
    ctx = torch.tensor(close[t - LOOKBACK : t + 1])  # 1D context
    with torch.no_grad():
        fc = pipeline.predict(ctx, prediction_length=PRED_LEN)  # [1, Q, 1]
    yhat = get_median_forecast(fc, pipeline)
    pred_next[t] = yhat

    today = close[t]
    if yhat > today * (1.0 + THRESH):
        signals[t] = 1
    elif yhat < today * (1.0 - THRESH):
        signals[t] = -1
    else:
        signals[t] = 0

In [9]:
df["Signal"] = signals
df["PredNextClose"] = pred_next

In [10]:
# backtesting
# -----------------------
class ChronosNextDayDirection(Strategy):
    def init(self):
        # backtesting.py позволяет обращаться к колонкам как self.data.<Name>
        pass

    def next(self):
        sig = int(self.data.Signal[-1])

        # если сигнала нет — выходим в кэш
        if sig == 0:
            if self.position:
                self.position.close()
            return

        # long
        if sig == 1:
            if self.position.is_short:
                self.position.close()
            if not self.position:
                self.buy()

        # short
        if sig == -1:
            if self.position.is_long:
                self.position.close()
            if not self.position:
                self.sell()

In [11]:
bt = Backtest(
    df, ChronosNextDayDirection,
    cash=100_000,
    commission=0.001,   # 0.1% комиссия (пример)
    trade_on_close=True # сделка по close текущего бара
)

In [12]:
stats = bt.run()
print(stats)
bt.plot()

Backtest.run:   0%|          | 0/1997 [00:00<?, ?bar/s]

/var/folders/lk/6f78t4jn60s5_ntqqc2dj0980000gn/T/ipykernel_27654/1138324275.py:1: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()


Start                     2018-01-02 00:00:00
End                       2025-12-11 00:00:00
Duration                   2900 days 00:00:00
Exposure Time [%]                    77.37738
Equity Final [$]                   98844.9897
Equity Peak [$]                   126593.1645
Commissions [$]                   38388.26821
Return [%]                           -1.15501
Buy & Hold Return [%]                545.6055
Return (Ann.) [%]                    -0.14642
Volatility (Ann.) [%]                26.52449
CAGR [%]                              -0.1009
Sharpe Ratio                         -0.00552
Sortino Ratio                        -0.00766
Calmar Ratio                         -0.00352
Alpha [%]                          -203.62713
Beta                                   0.3711
Max. Drawdown [%]                   -41.64167
Avg. Drawdown [%]                    -9.07713
Max. Drawdown Duration     1315 days 00:00:00
Avg. Drawdown Duration      133 days 00:00:00
# Trades                          

GridPlot(id='p1336', ...)

## Попробуем дообучить

In [13]:
SPLIT_DATE = "2025-01-01"

df_train = df.loc[df.index < SPLIT_DATE].copy()
df_test  = df.loc[df.index >= SPLIT_DATE].copy()

print(len(df_train), len(df_test))

1761 237


Chronos не дообучаем весами, используем один pipeline, но учим торговую логику на трейне.

In [14]:
def chronos_predict_next_close(df, pipeline, lookback=256):
    close = df["Close"].values.astype(np.float32)
    preds = np.full(len(df), np.nan)

    for t in range(lookback, len(df) - 1):
        ctx = torch.tensor(
            close[t - lookback : t + 1],
        )
        with torch.no_grad():
            fc = pipeline.predict(ctx, prediction_length=1)

        preds[t] = get_median_forecast(fc, pipeline)

    return preds

“Дообучение” на трейне (recalibration)

Мы учим порог сигнала на трейне — это и есть корректное fine-tuning стратегии.

1. Получаем прогнозы на трейне
2. Затем подбираем оптимальный threshold

Это ключевой момент - Мы не подгоняем модель — мы подгоняем decision rule.

In [15]:
df_train["PredNext"] = chronos_predict_next_close(
    df_train, pipeline, LOOKBACK
)

df_train = df_train.dropna(subset=["PredNext"])

In [16]:
returns = df_train["Close"].pct_change().shift(-1)

candidates = np.linspace(0.0, 0.01, 21)  # 0% … 1%
best_thr, best_sharpe = None, -np.inf

for thr in candidates:
    signal = np.where(
        df_train["PredNext"] > df_train["Close"] * (1 + thr),  1,
        np.where(df_train["PredNext"] < df_train["Close"] * (1 - thr), -1, 0)
    )

    strat_ret = signal * returns
    sharpe = strat_ret.mean() / strat_ret.std()

    if sharpe > best_sharpe:
        best_sharpe = sharpe
        best_thr = thr

print("Best threshold:", best_thr, "Sharpe:", best_sharpe)

Best threshold: 0.004 Sharpe: 0.03247839660163112


In [17]:
df_test["PredNext"] = chronos_predict_next_close(
    df_test, pipeline, LOOKBACK
)

df_test["Signal"] = np.where(
    df_test["PredNext"] > df_test["Close"] * (1 + best_thr),  1,
    np.where(df_test["PredNext"] < df_test["Close"] * (1 - best_thr), -1, 0)
)

In [18]:
class ChronosStrategy(Strategy):
    def init(self):
        # backtesting.py позволяет обращаться к колонкам как self.data.<Name>
        pass
    
    def next(self):
        sig = int(self.data.Signal[-1])

        if sig == 0:
            if self.position:
                self.position.close()
            return

        if sig == 1:
            if self.position.is_short:
                self.position.close()
            if not self.position:
                self.buy()

        if sig == -1:
            if self.position.is_long:
                self.position.close()
            if not self.position:
                self.sell()

In [19]:
bt_chronos = Backtest(
    df_test,
    ChronosStrategy,
    cash=100_000,
    commission=0.001,
    trade_on_close=True
)

stats_chronos = bt_chronos.run()
print(stats_chronos)

Backtest.run:   0%|          | 0/236 [00:00<?, ?bar/s]

Start                     2025-01-02 00:00:00
End                       2025-12-11 00:00:00
Duration                    343 days 00:00:00
Exposure Time [%]                         0.0
Equity Final [$]                     100000.0
Equity Peak [$]                      100000.0
Return [%]                                0.0
Buy & Hold Return [%]                14.01681
Return (Ann.) [%]                         0.0
Volatility (Ann.) [%]                     0.0
CAGR [%]                                  0.0
Sharpe Ratio                              NaN
Sortino Ratio                             NaN
Calmar Ratio                              NaN
Alpha [%]                                 0.0
Beta                                      0.0
Max. Drawdown [%]                        -0.0
Avg. Drawdown [%]                         NaN
Max. Drawdown Duration                    NaN
Avg. Drawdown Duration                    NaN
# Trades                                    0
Win Rate [%]                      

In [20]:
bt.plot()

GridPlot(id='p1693', ...)

In [21]:
class BuyHold(Strategy):
    def init(self):
        # backtesting.py позволяет обращаться к колонкам как self.data.<Name>
        pass
    
    def next(self):
        if not self.position:
            self.buy()

In [22]:
bt_bh = Backtest(
    df_test,
    BuyHold,
    cash=100_000,
    commission=0.001,
    trade_on_close=True
)

stats_bh = bt_bh.run()
print(stats_bh)

Backtest.run:   0%|          | 0/236 [00:00<?, ?bar/s]

Start                     2025-01-02 00:00:00
End                       2025-12-11 00:00:00
Duration                    343 days 00:00:00
Exposure Time [%]                         0.0
Equity Final [$]                 114114.92165
Equity Peak [$]                  117460.52315
Return [%]                           14.11492
Buy & Hold Return [%]                14.01681
Return (Ann.) [%]                    15.07254
Volatility (Ann.) [%]                39.22851
CAGR [%]                             10.18669
Sharpe Ratio                          0.38422
Sortino Ratio                         0.68922
Calmar Ratio                          0.49932
Alpha [%]                             0.11972
Beta                                  0.99846
Max. Drawdown [%]                   -30.18605
Avg. Drawdown [%]                    -5.13753
Max. Drawdown Duration      210 days 00:00:00
Avg. Drawdown Duration       30 days 00:00:00
# Trades                                    0
Win Rate [%]                      

/var/folders/lk/6f78t4jn60s5_ntqqc2dj0980000gn/T/ipykernel_27654/4044010920.py:9: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats_bh = bt_bh.run()


In [23]:
comparison = pd.DataFrame({
    "Chronos": stats_chronos,
    "Buy&Hold": stats_bh
})

comparison

,Chronos,Buy&Hold
Start,2025-01-02 00:00:00,2025-01-02 00:00:00
End,2025-12-11 00:00:00,2025-12-11 00:00:00
Duration,343 days 00:00:00,343 days 00:00:00
Exposure Time [%],0.0,0.0
Equity Final [$],100000.0,114114.921649
Equity Peak [$],100000.0,117460.52315
Return [%],0.0,14.114922
Buy & Hold Return [%],14.01681,14.01681
Return (Ann.) [%],0.0,15.072541
Volatility (Ann.) [%],0.0,39.228514
